# Checkpoint 4 | Fine-tuning de AG News con LoRA

Se entrena DistilBERT para clasificar cuatro categorías de AG News y se comparan las métricas con TF-IDF. Se utiliza GPU en Google Colab.


In [ ]:
import torch
assert torch.cuda.is_available(), "Seleccioná T4 GPU en el entorno de Colab antes de continuar."
print("GPU disponible:", torch.cuda.get_device_name(0))


## 1. Carga del repositorio


In [ ]:
import subprocess
from pathlib import Path

ROOT = Path("/content/data-science-iii-agnews")
if ROOT.exists():
    subprocess.check_call(["git", "-C", str(ROOT), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "https://github.com/h1ghsv/data-science-iii-agnews.git", str(ROOT)])
assert (ROOT / "checkpoint_4_lora" / "entrenar_lora.py").exists(), "No se encontró el código del Checkpoint 4 en el repositorio."
print("Repositorio preparado:", ROOT)


## 2. Dependencias


In [ ]:
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "checkpoint_4_lora" / "requirements_lora.txt")])
print("Dependencias instaladas. GPU CUDA:", torch.cuda.is_available())


## 3. Pruebas del pipeline


In [ ]:
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "checkpoint_4_lora/tests"], cwd=ROOT)


## 4. Entrenamiento y validación

La opción `REFIT_COMPLETO=True` permite entrenar el modelo final con los 8.000 documentos tras seleccionar la mejor época en validación.


In [ ]:
REFIT_COMPLETO = True
cmd = [sys.executable, "-u", "-m", "checkpoint_4_lora.entrenar_lora",
       "--epochs", "3", "--max-length", "128", "--batch-size", "16",
       "--r", "8", "--alpha", "16", "--lr", "0.0001"]
if REFIT_COMPLETO:
    cmd.append("--refit-full")
subprocess.check_call(cmd, cwd=ROOT)


## 5. Informe técnico


In [ ]:
subprocess.check_call([sys.executable, "-m", "checkpoint_4_lora.generar_pdf"], cwd=ROOT)
pdf = ROOT / "checkpoint_4_lora" / "reportes" / "Arevalo_JuanPablo_Checkpoint_NLP3.pdf"
assert pdf.exists() and pdf.stat().st_size < 10*1024*1024
print("PDF listo:", pdf.name, "-", round(pdf.stat().st_size/1024), "KB")


## 6. Resultados

Se descargan el PDF y los archivos de métricas.


In [ ]:
import zipfile
from google.colab import files

destino_zip = Path("/content/AGNews_C4_Resultados_Reales.zip")
with zipfile.ZipFile(destino_zip, "w", zipfile.ZIP_DEFLATED) as z:
    carpeta = ROOT / "checkpoint_4_lora"
    for archivo in sorted((carpeta / "outputs").glob("*")):
        if archivo.is_file():
            z.write(archivo, archivo.relative_to(ROOT))
    z.write(pdf, pdf.relative_to(ROOT))
files.download(str(pdf))
files.download(str(destino_zip))
